# notebook_dna_gold.ipynb

Gold-layer DNA tables.

## Cell 1 — `gold_person_dna`
One row per person x DNA citation. Joins GEDCOM tags + person-root citations + scraped match data + branch + proximity.

## Cell 2 — `gold_dna_coverage`
Per-branch DNA coverage summary. **Updated Session 44: 20cM threshold** (was 100cM).

## Run order
1. Run Cell 1
2. Run Cell 2

## Cell 1 — `gold_person_dna`
Unchanged from Session 42.

In [0]:
# gold_person_dna — one row per person x DNA citation
# Sources: GEDCOM DNA tags + person-root source citations + scraped match data
#          + branch assignment + ancestral proximity
#
# LEFT joins throughout so:
#   - Tagged people without citations (e.g. Common DNA Ancestors) are retained
#   - People without a scraped row are retained (NULL scraped fields)

spark.sql("""
CREATE OR REPLACE TABLE genealogy.gold_person_dna AS

WITH

-- All people with any DNA tag
dna_tags AS (
  SELECT
    pt.person_gedcom_id,
    t.tag_name AS dna_role
  FROM genealogy.silver_person_tag pt
  JOIN genealogy.silver_tag t
    ON t.tag_gedcom_id = pt.tag_gedcom_id
  WHERE t.tag_name IN ('DNA Match', 'Common DNA Ancestor', 'DNA Connection')
),

-- Kit source xref -> kit_id mapping
kit_map AS (
  SELECT '@S1272999949@' AS source_xref, 'ed_ancestry'      AS kit_id UNION ALL
  SELECT '@S1275055303@',                'gillian_ancestry'            UNION ALL
  SELECT '@S1275093887@',                'dad_ancestry'
),

-- Person-root Ancestry DNA citations
dna_citations AS (
  SELECT
    ps.person_gedcom_id,
    km.kit_id,
    REGEXP_REPLACE(ps.citation_page, '^AncestryDNA Match to ', '') AS match_name,
    CAST(
      REGEXP_EXTRACT(ps.citation_text, 'sharing ([0-9]+\\.?[0-9]*) cM', 1)
    AS DOUBLE) AS citation_cm,
    CAST(
      REGEXP_EXTRACT(ps.citation_text, 'across ([0-9]+) segment', 1)
    AS INT) AS citation_segments,
    REGEXP_EXTRACT(ps.citation_text, 'Predicted Relation: ([^,]+)', 1)
      AS citation_predicted_relationship,
    REGEXP_REPLACE(ps.citation_date, '^Accessed: ', '') AS citation_access_date,
    ps.citation_url,
    ps.citation_note
  FROM genealogy.silver_person_source ps
  JOIN kit_map km ON km.source_xref = ps.source_xref
)

SELECT
  t.person_gedcom_id,
  t.dna_role,
  c.kit_id,
  c.match_name,
  c.citation_cm,
  c.citation_segments,
  c.citation_predicted_relationship,
  c.citation_access_date,
  c.citation_url,
  c.citation_note,
  s.shared_cm        AS scraped_cm,
  s.side,
  s.has_common_ancestor_hint,
  s.tree_info,
  s.tree_people_count,
  s.active,
  s.first_seen_date,
  s.last_seen_date,
  b.branch,
  p.ancestral_proximity

FROM dna_tags t
LEFT JOIN dna_citations      c ON c.person_gedcom_id = t.person_gedcom_id
LEFT JOIN genealogy.silver_dna_match s
  ON s.kit_id = c.kit_id AND s.match_name = c.match_name
LEFT JOIN genealogy.gold_person_branch  b ON b.person_gedcom_id = t.person_gedcom_id
LEFT JOIN genealogy.gold_ancestral_proximity p ON p.person_id = t.person_gedcom_id
""")

print('gold_person_dna: OK')
spark.sql('SELECT dna_role, COUNT(*) AS n FROM genealogy.gold_person_dna GROUP BY dna_role ORDER BY n DESC').display()


## Cell 2 — `gold_dna_coverage`

**Updated Session 44: 20cM threshold** (was 100cM).

Column definitions:
- `close_matches_total` — active scraped matches with `shared_cm >= 20` linked to this branch
- `unlinked_close_matches_total` — active scraped matches with `shared_cm >= 20` with no `gold_person_dna` row (cross-branch scalar)

In [0]:
# gold_dna_coverage — per-branch DNA coverage summary
# Threshold: 20cM throughout (updated Session 44, was 100cM)
#
# close_matches_total uses silver_dna_match.shared_cm (scraped value)
# for consistency with unlinked_close_matches_total.

spark.sql("""
CREATE OR REPLACE TABLE genealogy.gold_dna_coverage AS

WITH

-- Direct ancestor counts per branch (proximity = 0)
ancestor_base AS (
  SELECT
    b.branch,
    COUNT(DISTINCT p.person_id)                            AS direct_ancestors_total,
    COUNT(DISTINCT CASE
      WHEN g.dna_role = 'Common DNA Ancestor'
      THEN p.person_id END)                               AS direct_ancestors_dna
  FROM genealogy.gold_ancestral_proximity p
  JOIN genealogy.gold_person_branch b
    ON b.person_gedcom_id = p.person_id
  LEFT JOIN genealogy.gold_person_dna g
    ON g.person_gedcom_id = p.person_id
  WHERE p.ancestral_proximity = 0
  GROUP BY b.branch
),

-- DNA Match counts per branch
match_counts AS (
  SELECT
    branch,
    COUNT(DISTINCT person_gedcom_id)                       AS dna_matches_total,
    COUNT(DISTINCT CASE WHEN active = TRUE
      THEN person_gedcom_id END)                          AS dna_matches_active
  FROM genealogy.gold_person_dna
  WHERE dna_role = 'DNA Match'
  GROUP BY branch
),

-- Close matches per branch: active scraped >= 20cM
-- Joined via gold_person_dna to get branch assignment.
-- COUNT(DISTINCT match_name) avoids double-counting a match appearing in
-- multiple kits that happens to link to the same branch.
close_by_branch AS (
  SELECT
    g.branch,
    COUNT(DISTINCT s.match_name)                           AS close_matches_total
  FROM genealogy.silver_dna_match s
  JOIN genealogy.gold_person_dna g
    ON g.kit_id = s.kit_id AND g.match_name = s.match_name
  WHERE s.active = TRUE
    AND s.shared_cm >= 20
    AND g.branch IS NOT NULL
  GROUP BY g.branch
),

-- Unlinked close matches: active scraped >= 20cM with no gold_person_dna row.
-- Cross-branch scalar — same value repeated on all branch rows.
unlinked AS (
  SELECT COUNT(*) AS unlinked_close_matches_total
  FROM genealogy.silver_dna_match s
  LEFT JOIN genealogy.gold_person_dna g
    ON g.kit_id = s.kit_id AND g.match_name = s.match_name
  WHERE s.active = TRUE
    AND s.shared_cm >= 20
    AND g.person_gedcom_id IS NULL
),

-- All known branches as spine so 0-count branches are not silently dropped
branches AS (
  SELECT DISTINCT branch
  FROM genealogy.gold_person_branch
  WHERE branch IS NOT NULL
)

SELECT
  br.branch,
  COALESCE(ab.direct_ancestors_total, 0)           AS direct_ancestors_total,
  COALESCE(ab.direct_ancestors_dna,   0)           AS direct_ancestors_dna,
  CASE
    WHEN COALESCE(ab.direct_ancestors_total, 0) = 0 THEN 0.0
    ELSE ROUND(
      ab.direct_ancestors_dna * 100.0 / ab.direct_ancestors_total, 1
    )
  END                                              AS ancestor_dna_pct,
  COALESCE(mc.dna_matches_total,  0)               AS dna_matches_total,
  COALESCE(mc.dna_matches_active, 0)               AS dna_matches_active,
  COALESCE(cb.close_matches_total, 0)              AS close_matches_total,
  ul.unlinked_close_matches_total

FROM branches br
LEFT JOIN ancestor_base   ab ON ab.branch = br.branch
LEFT JOIN match_counts    mc ON mc.branch = br.branch
LEFT JOIN close_by_branch cb ON cb.branch = br.branch
CROSS JOIN unlinked ul

ORDER BY ancestor_dna_pct DESC
""")

print('gold_dna_coverage: OK')
spark.sql('SELECT * FROM genealogy.gold_dna_coverage ORDER BY ancestor_dna_pct DESC').display()


## Cell 3 — Verification

In [0]:
# Confirm jeanelliott79 still parses correctly
spark.sql("""
  SELECT person_gedcom_id, dna_role, kit_id, match_name,
         citation_cm, citation_segments, citation_predicted_relationship
  FROM genealogy.gold_person_dna
  WHERE match_name = 'jeanelliott79'
""").display()
# Expected: 1 row for gillian_ancestry (Ed citation still missing from tree)

# Coverage with updated 20cM threshold
spark.sql("""
  SELECT branch, ancestor_dna_pct, close_matches_total, unlinked_close_matches_total
  FROM genealogy.gold_dna_coverage
  ORDER BY ancestor_dna_pct DESC
""").display()

# Sanity: sum of close_matches_total across branches should be <= active scraped >= 20cM
# (<= because matches with NULL branch are excluded from branch rows)
spark.sql("""
  SELECT COUNT(*) AS active_close_scraped_total
  FROM genealogy.silver_dna_match
  WHERE active = TRUE AND shared_cm >= 20
""").display()
